# EXP-005: Random Forest mặc định

- Ngày: 22/11/2026 (tổng hợp notebook).
- Ngày chạy: 16/11/2026. kiểm tra chạy lại 18/11/2026.
- Mã thực nghiệm: exp005.
- So với lần trước: G3 (giữ RF mặc định, kiểm tra chạy lại).

**Thực nghiệm tương ứng:** [experiment.ipynb](../../experiments/01_baselines/EXP-005_RF/experiment.ipynb).

**Mạch thực nghiệm:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).


## 1. Mục tiêu và giả thuyết

**Mục tiêu:** Chạy RF ban đầu để có điểm so sánh với các cách cải thiện sau này.

**Giả thuyết:** Kỳ vọng RF học từ 68 đặc trưng sẽ tốt hơn Dummy. Cũng kỳ vọng chạy lại cùng cấu hình sẽ khớp kết quả G3.


## 2. Cấu hình

Dùng cấu hình trong `experiment.ipynb` và cấu hình đã lưu khi chạy.

Dùng RF với 100 cây, `max_depth=None`, `max_features="sqrt"`, `class_weight=None`, seed 42, `n_jobs=1`. dùng đủ 68 đặc trưng. Điền giá trị thiếu bằng trung vị tính từ train.

**Thay đổi so với G3:** không đổi tham số. Chạy lại để kiểm tra và so sánh kết quả.


## 3. Cách thực hiện

Dùng dữ liệu physiological68 gồm 6.189 cửa sổ, 68 đặc trưng và 15 người. Bài toán nhị phân phân biệt không stress/stress. Bài toán ba lớp gồm baseline/stress/amusement.

Chạy RF mặc định bằng LOSO. Cho mô hình học từ 14 người và kiểm tra trên 1 người. Giữ riêng từng người cho đến khi kiểm tra đủ 15 người. Hai bài toán có 30 fold. Ghép kết quả với G3 theo task, người và mã cửa sổ.

Chỉ dùng train để học cách xử lý dữ liệu. Tính Accuracy, Macro-F1 và Balanced Accuracy cho từng người test. Sau đó lấy trung bình đều qua 15 người.


## 4. Kết quả

Accuracy là tỷ lệ dự đoán đúng. Macro-F1 là trung bình F1 các lớp. Balanced Accuracy là trung bình recall các lớp. 

| Bài toán | Phiên bản | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|
| binary | rf | 0.8900 | 0.8735 | 0.8865 |
| multiclass | rf | 0.7717 | 0.6480 | 0.6700 |


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

here = Path.cwd()
repo = next(p for p in (here, *here.parents) if (p / 'src/wesad/evaluation.py').is_file())
source = repo / 'outputs/exp005_rf_full'
file = source / ('results.csv' if (source / 'results.csv').exists() else 'metrics_by_subject.csv')
scores = pd.read_csv(file)
if 'model' in scores:
    scores = scores.loc[scores.model.eq('rf')]
if 'variant' not in scores:
    scores['variant'] = 'rf'
keys = ['task', 'variant']
display(scores.groupby(keys)[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(4))


macro_f1         balanced_accuracy        
                       mean     std              mean     std
task       variant                                           
binary     rf        0.8735  0.1440            0.8865  0.1143
multiclass rf        0.6480  0.1297            0.6700  0.1204

### RF và các mô hình cơ sở

![RF và các mô hình cơ sở](../../experiments/01_baselines/EXP-005_RF/figures/rf_vs_baselines_g3.png)


### Train và test

![Train và test](../../experiments/01_baselines/EXP-005_RF/figures/train_vs_test.png)


## 5. So sánh với các lần trước

Dùng điểm test trong bảng. Chỉ so sánh các kết quả trong cùng bài toán.

| Bài toán | Lần chạy | Thay đổi | Accuracy | F1 (macro) | Balanced Accuracy |
|---|---|---|---|---|---|
| binary | G3 RF | RF mặc định đã lưu | 0.8900 | 0.8735 | 0.8865 |
| multiclass | G3 RF | RF mặc định đã lưu | 0.7717 | 0.6480 | 0.6700 |
| binary | exp005, chạy lại | Không đổi tham số | 0.8900 | 0.8735 | 0.8865 |
| multiclass | exp005, chạy lại | Không đổi tham số | 0.7717 | 0.6480 | 0.6700 |


## 6. Nhận xét

- **Giả thuyết:** thấy RF vượt Dummy về Macro-F1 và BA ở cả hai bài toán. Hai chỉ số khớp G3 ở đủ 30 fold. 12.378 dự đoán không đổi nhãn.
- **Quan sát bất thường:** không thấy điểm khác nhau giữa hai lần so sánh. Bài toán ba lớp có điểm thấp hơn nhị phân. Không tính hai lần chạy khớp nhau thành hai tập kiểm tra độc lập.
- **Lỗi / khó khăn:** kết quả ban đầu chưa có điểm train. Fit lại cấu hình cũ để tính thêm điểm train. Kiểm tra dự đoán test trước khi lưu.


## 7. Bước tiếp theo

1. Giữ RF mặc định làm mốc so sánh.
2. Sang EXP-009 để thử chỉnh tham số trong train.
3. Giữ cùng dữ liệu và LOSO để so sánh các thực nghiệm.

**Liên kết:** [EXP-005](Mai_EXP-005-RF.ipynb) → [EXP-009](Mai_EXP-009_RF_tuning.ipynb) → [EXP-014](Mai_EXP-014_RF_balancing.ipynb) → [EXP-019](Mai_EXP-019_RF_.ipynb).
